In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\akash\AppData\Local\Temp\ipykernel_20328\170880071.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\akash\OneDrive\Desktop\finance_analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

In [2]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.retrievers import BM25Retriever

In [32]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [3]:
def load_financial_report(file_path: str):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

In [4]:
docs = load_financial_report(
    "data/reports/Apple_2025_Annual_Report.pdf"
)

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [6]:
def split_documents(documents):

    chunks = text_splitter.split_documents(documents)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    return chunks

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3069.21it/s]


In [8]:
chunks = split_documents(docs)


In [9]:

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)


In [38]:
base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 8}
)

In [25]:
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True
)

In [18]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [21]:
from collections import defaultdict

def reciprocal_rank_fusion(retriever_results, k=60):
    """
    Combine results from multiple retrievers using Reciprocal Rank Fusion (RRF).

    Args:
        retriever_results: List of lists of Documents
        k: RRF constant (default = 60)

    Returns:
        List[Document] sorted by fused score
    """

    scores = defaultdict(float)
    unique_docs = {}

    for docs in retriever_results:
        for rank, doc in enumerate(docs):

            # Unique document ID
            doc_id = (
                doc.metadata.get("source", "")
                + "_"
                + str(doc.metadata.get("chunk_id", rank))
            )

            unique_docs[doc_id] = doc

            scores[doc_id] += 1 / (k + rank + 1)

    ranked_docs = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        unique_docs[doc_id]
        for doc_id, _ in ranked_docs
    ]

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
def compress_documents(query, documents):

    compressed_docs = compressor.compress_documents(
        documents=documents,
        query=query
    )

    return compressed_docs

In [54]:
def hybrid_retrieve(query, k=8):

    semantic_docs = retriever.invoke(query)

    keyword_docs = bm25_retriever.invoke(query)

    fused_docs = reciprocal_rank_fusion(
        [
            semantic_docs,
            keyword_docs
        ]
    )

    # Take only the top-k fused documents
    fused_docs = fused_docs[:k]

    compressed_docs = compress_documents(
        query,
        fused_docs
    )

    return compressed_docs

In [27]:
query = "What are Apple's biggest business risks?"

docs = hybrid_retrieve(query)

COMPRESS FUNCTION

In [48]:
from pydantic import BaseModel
from typing import Literal

class SelfRAGDecision(BaseModel):
    decision: Literal["SUFFICIENT", "INSUFFICIENT"]
    reason: str


In [49]:
self_rag_llm = llm.with_structured_output(SelfRAGDecision)

In [55]:
def generate_answer(query, k=8):

    docs = hybrid_retrieve(query, k)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
You are a Senior Financial Research Analyst.

Answer ONLY using the provided context.

Context:
{context}

Question:
{query}
"""

    answer = llm.invoke(prompt)

    return answer, context

In [51]:
def evaluate_answer(question, context, answer):

    prompt = f"""
You are evaluating a Retrieval-Augmented Generation (RAG) answer.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Determine whether the retrieved context is sufficient to support the answer.

Rules:

1. If the answer is fully supported by the context,
return SUFFICIENT.

2. If important information is missing,
return INSUFFICIENT.

3. Ignore writing quality.

4. Evaluate ONLY factual support.

Provide a short reason.
"""

    return self_rag_llm.invoke(prompt)

In [62]:
def self_rag(query):

    print("=" * 80)
    print("FIRST RETRIEVAL")
    print("=" * 80)

    answer, context = generate_answer(query, k=8)

    evaluation = evaluate_answer(
        query,
        context,
        answer.content
    )

    print(f"Decision : {evaluation.decision}")
    print(f"Reason   : {evaluation.reason}")

    # Enough evidence
    if evaluation.decision == "SUFFICIENT":

        return {
            "answer": answer.content,
            "decision": evaluation.decision,
            "reason": evaluation.reason,
            "retrievals": 1,
            "rewritten_query": None
        }

    print("\nRewriting query...\n")

    rewritten = rewrite_query(
        query,
        evaluation.reason
    )

    print(f"New Query : {rewritten.rewritten_query}")

    print("\nRetrieving again...\n")

    answer, context = generate_answer(
        rewritten.rewritten_query,
        k=12
    )

    return {
        "answer": answer.content,
        "decision": "SUFFICIENT_AFTER_RETRY",
        "reason": evaluation.reason,
        "retrievals": 2,
        "rewritten_query": rewritten.rewritten_query
    }

In [58]:
query = "Summarize Apple's annual report."

final_answer = self_rag(query)

print(final_answer)

Decision: SUFFICIENT
Reason: The retrieved context provides all necessary financial information and market value data to support the generated answer.
Based on the provided information, here's a summary of Apple's annual report:

**Key Financial Highlights:**

- Net sales: $416,161 (latest year), $391,035 (previous year), $383,285 (year before previous)
- Gross margin: $195,201 (latest year), $180,683 (previous year), $169,148 (year before previous)
- Operating income: $133,050 (latest year), $123,216 (previous year), $114,301 (year before previous)
- Net income: $112,010 (latest year), $93,736 (previous year), $96,995 (year before previous)

**Market Value:**

- The aggregate market value of the voting and non-voting stock held by non-affiliates of Apple Inc. as of March 28, 2025, was approximately $3,253,431,000,000.


In [59]:
from pydantic import BaseModel

class QueryRewrite(BaseModel):
    rewritten_query: str

In [60]:
rewrite_llm = llm.with_structured_output(QueryRewrite)

In [61]:
def rewrite_query(question, evaluation_reason):

    prompt = f"""
You are improving a search query for a financial RAG system.

Original Question:
{question}

The evaluator said:

{evaluation_reason}

Rewrite the query so it retrieves more relevant
information from a company's annual report.

Use financial terminology.

Return only the rewritten query.
"""

    return rewrite_llm.invoke(prompt)

In [63]:
query = "Tell me about Apple's future."

result = self_rag(query)

print("=" * 80)
print(result["answer"])

print("=" * 80)
print(result["decision"])

print(result["rewritten_query"])

FIRST RETRIEVAL
Decision : INSUFFICIENT
Reason   : The retrieved context only provides information about Apple's product and software announcements in fiscal year 2025, but does not provide any information about Apple's future plans beyond that year.

Rewriting query...

New Query : What are Apple's long-term strategic objectives and growth prospects outlined in their annual report?

Retrieving again...

Based on the provided context, Apple's long-term strategic objectives and growth prospects can be inferred as follows:

1. **Continuing Innovation**: The Company aims to introduce innovative new products, services, and technologies to the marketplace, ensuring its competitiveness.
2. **Market Expansion**: Apple is focused on expanding its market opportunities in various product categories, including:
	* Smartphones
	* Personal computers
	* Tablets
	* Wearables
3. **Competitive Advantage**: The Company seeks to maintain a strong competitive position through:
	* Design and technology inn